In [0]:
%sql
    -- restaurant_id STRING PRIMARY KEY,
    -- restaurant_name STRING,
    -- city STRING,
    -- total_reviews BIGINT,
    -- avg_rating DECIMAL(3,2),
    -- rating_5_count BIGINT,
    -- rating_4_count BIGINT,
    -- rating_3_count BIGINT,
    -- rating_2_count BIGINT,
    -- rating_1_count BIGINT,
    -- sentiment_positive_count BIGINT,
    -- sentiment_neutral_count BIGINT,
    -- sentiment_negative_count BIGINT

In [0]:
from pyspark.sql.functions import *

In [0]:
df_review_stats=(
    spark.table("ws_dbxproject.`02_silver`.fact_reviews")
    .groupBy("restaurant_id")
    .agg(
        countDistinct("review_id").alias("total_reviews"),
        round(avg("rating"),2).alias("avg_rating"),
        sum(when(col("rating")==5,1).otherwise(0)).alias("rating_5_count"),
        sum(when(col("rating")==4,1).otherwise(0)).alias("rating_4_count"),
        sum(when(col("rating")==3,1).otherwise(0)).alias("rating_3_count"),
        sum(when(col("rating")==2,1).otherwise(0)).alias("rating_2_count"),
        sum(when(col("rating")==1,1).otherwise(0)).alias("rating_1_count"),
        sum(when(col("sentiment")=="positive",1).otherwise(0)).alias("sentiment_positive_count"),
        sum(when(col("sentiment")=="negative",1).otherwise(0)).alias("sentiment_negative_count"),
        sum(when(col("sentiment")=="neutral",1).otherwise(0)).alias("sentiment_neutral_count")
    )
)
display(df_review_stats)

In [0]:
df_restaurant=spark.table("ws_dbxproject.`02_silver`.dim_restaurants")

df_restaurant_reviews=(
    df_restaurant.join(df_review_stats, on ="restaurant_id", how="left")
    .select(
        "restaurant_id",
        col("name").alias("restaurant_name"),
        "city",
        coalesce(col("total_reviews"),lit(0)).alias("total_reviews"),
        coalesce(col("avg_rating"),lit(0)).alias("avg_rating"),
        coalesce(col("rating_5_count"),lit(0)).alias("rating_5_count"),
        coalesce(col("rating_4_count"),lit(0)).alias("rating_4_count"),
        coalesce(col("rating_3_count"),lit(0)).alias("rating_3_count"),
        coalesce(col("rating_2_count"),lit(0)).alias("rating_2_count"),
        coalesce(col("rating_1_count"),lit(0)).alias("rating_1_count"),
        coalesce(col("sentiment_positive_count"),lit(0)).alias("sentiment_positive_count"),
        coalesce(col("sentiment_neutral_count"),lit(0)).alias("sentiment_neutral_count"),
        coalesce(col("sentiment_negative_count"),lit(0)).alias("sentiment_negative_count")
    )
)
display(df_restaurant_reviews)